# Segmentation Measurement

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# statsmodels and pasty are used for linear regression and ANOVA
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices

import warnings
warnings.filterwarnings('ignore')

# Vessel Measurements

In [ ]:
df_vessel = pd.read_csv("path/to/quantification-notebooks/vessel_metrics.csv")
df_vessel.head()

### set filename as index

In [ ]:
df_vessel.index = df_vessel['filename'].str.split('_', expand=True)[1]
df_vessel

In [ ]:
df_vessel.columns[:60]

### Quick EDA

In [ ]:
def analyze_metrics(df, method='lee', drop_cols=None, figsize=(15, 8)):
    """
    Filter, clean, and visualize metrics

    Parameters
    ----------
    df : pd.DataFrame
        Raw  dataframe.
    method : str
        'lee','teasar' or 'pore_throat', selects which metric columns to keep.
    drop_median_cols : list[str], optional
        Extra columns to drop before correlation/pairplot (e.g. redundant medians).
        If None, no extra columns are dropped.
    figsize : tuple
        Size of the correlation heatmap.

    Returns
    -------
    metrics_coi_df : pd.DataFrame
        The filtered "columns of interest" dataframe (with 'index' col added).
    correlations : pd.DataFrame
        Correlation matrix used in the heatmap.
    """
    if method == 'lee':
        other_method = 'teasar'
    elif method == 'teasar':
        other_method = 'lee'
    else:
        other_method = 'pore_throat'
    
    # Drop voxel-scale columns (suffix "_vx") -- we want physical units (e.g. um), not voxel counts
    no_vx = [col for col in df.columns if 'vx' not in col.split('_')]
    metrics_no_vx = df[no_vx].copy()

    # Drop known non-metric columns (image metadata, not measurements)
    drop_cols = ['original_image_shape', 'crop', 'crop_shape', 'prop_of_1000cube', 'conv_factor']
    metrics_no_vx.drop([c for c in drop_cols if c in metrics_no_vx.columns], axis=1, inplace=True)
    if 'filename' in metrics_no_vx.columns:
        metrics_no_vx = metrics_no_vx.drop('filename', axis=1)

    # Keep only columns for the requested method (drop the columns that belong to the *other* skeletonizer)
    method_cols = [col for col in metrics_no_vx.columns if col.split('_')[0] not in other_method]
    metrics_method_df = metrics_no_vx[method_cols]

    # Keep only "columns of interest" 
    stat_suffixes = ['mean', 'std', 'dev', 'max', 'min', 'q1', 'q3', 'iqr']
    coi_cols = [col for col in metrics_method_df.columns if col.split('_')[-1] not in stat_suffixes]
    metrics_coi_df = metrics_method_df[coi_cols]

    # Optionally drop redundant columns before correlating
    if drop_cols:
        metrics_coi_df = metrics_coi_df[[c for c in metrics_coi_df.columns if c not in drop_cols]]

    # drop the last column before correlating -- assumes it's a non-metric column (e.g. surface-area
    # duplicate/id-like field); adjust this if your column ordering differs
    correlations = metrics_coi_df.iloc[:, :-1].corr()

    # Heatmap
    plt.figure(figsize=figsize)
    sns.heatmap(correlations, cmap='coolwarm_r', linewidths=2.0, annot=True)
    plt.title(f'{method.capitalize()} metric correlations', fontsize=16)
    plt.show()

    # Pairplots, colored by row index
    metrics_coi_df = metrics_coi_df.copy()
    metrics_coi_df['index'] = metrics_coi_df.index  # add sample ID as its own column so we can color by it
    metrics_coi_df = metrics_coi_df.rename(columns={'surface-area': 'surface_area'})  # Rename for consistency
    x_vars = metrics_coi_df.columns[:-2]  # exclude 'index' and target col used for pairing

    # one PairGrid per metric: each metric plotted against every other metric, colored by sample
    for y in x_vars:
        g = sns.PairGrid(metrics_coi_df, hue="index", x_vars=x_vars, y_vars=[y], palette='colorblind')
        g.map_diag(sns.histplot, hue=None)
        g.map_offdiag(sns.scatterplot, s=70)
        g.add_legend()
        g.fig.suptitle(f' {y} vs other metrics', y=1.02)
        plt.show()

    return metrics_coi_df, correlations

### Lee Measurement

In [ ]:
metrics_lee_df_vessel, lee_correlations_vessel = analyze_metrics(
    df_vessel,
    method='lee',
    drop_cols=['lee_ba_median', 'lee_ba3d_median']
)

### Teasar Measurement

In [ ]:
metrics_teasar_df_vessel, teasar_correlations_vessel = analyze_metrics(
    df_vessel,
    method='teasar',
    drop_cols=['teasar_ba_median', 'teasar_ba3d_median']
)

### Correlation Scatter + Regression Plot
Scatter plot with hue grouping alongside an OLS regression fit, side by side, for two chosen metrics.

In [ ]:
def plot_correlation_pair(df, colxn, colyn, hue_col='index', n_colors=3,
                           save_path=None, figsize=(7, 4)):
    """
    Plot a scatter (colored by hue_col) and an OLS regression fit,
    side by side, for two columns in df.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe containing the two metric columns and the hue column.
    colxn : str
        Name of the X-axis column.
    colyn : str
        Name of the Y-axis column.
    hue_col : str
        Column to color the scatter points by.
    n_colors : int
        Number of distinct colors for the hue palette.
    save_path : str, optional
        If provided, saves the figure to this path (png, 300 dpi).
    figsize : tuple
        Figure size.

    Returns
    -------
    res : statsmodels regression results object
        Fitted OLS model, in case you want to inspect it further.
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    ax1, ax2 = axes[0], axes[1]

    colx = df[colxn]
    coly = df[colyn]
    ttxt = f"{colxn} vs {colyn}"
    fig.suptitle(ttxt)

    # dmatrices builds the design matrices from a patsy formula string, e.g. "colxn ~ colyn"
    vars_ = [colxn, colyn]
    df_vars = df[vars_]
    regln = f"{colxn} ~ {colyn}"
    y, x = dmatrices(regln, df_vars, return_type='dataframe')
    mod = sm.OLS(y, x)
    results = mod.fit()
    print(results.summary())
    r2num = results.rsquared

    # Scatter plot 
    palette_ = sns.husl_palette(n_colors=n_colors, h=0.01, s=0.9, l=0.6, as_cmap=False)
    sns.scatterplot(df, x=colx, y=coly, hue=hue_col, palette=palette_,
                     s=70, legend=True, ax=ax1)

    # Regression plot
    sns.regplot(df, x=colx, y=coly, color='black', marker='x', ax=ax2)
    ax2.set_title(f"R-Squared = {r2num}")

    plt.tight_layout()

    # if save_path:
    #     plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.5)

    plt.show()
    return results

In [ ]:
# example usage: check whether skeleton length tracks surface area for the Lee method
vessel_results = plot_correlation_pair(
    metrics_lee_df_vessel,
    colxn="lee_skeleton_length",
    colyn="surface_area",
    n_colors=3
)

In [ ]:
# same comparison, but using Teasar skeleton length instead
vessel_results = plot_correlation_pair(
    metrics_teasar_df_vessel,
    colxn="teasar_skeleton_length",
    colyn="surface_area",
    n_colors=3
)

In [ ]:
df_vessel['sample_id']= df_vessel['filename'].str.split('_', expand=True)[1]
plt.figure(figsize=(8, 5))
sns.barplot(x=df_vessel['sample_id'], y=df_vessel['surface-area'], hue=df_vessel['sample_id'], palette='colorblind')
plt.xlabel("Sample ID")
plt.ylabel("Surface Area")
plt.title("Vessel Surface Area by Sample ID")
plt.tight_layout();

# IVS (Intervillous  space) Measurements

In [ ]:
# intervillous space (IVS) is the pore network between villi 
df_ivs = pd.read_csv("/path/to/quantification-notebooks/ivs_metrics.csv")
df_ivs.head()

### set filename as index

In [ ]:
df_ivs.index = df_ivs['filename'].str.split('_', expand=True)[1]
df_ivs

### Quick EDA

In [ ]:
df_ivs.columns

In [ ]:
metrics_df_ivs, correlations_ivs = analyze_metrics(
    df_ivs,
    method='pore_throat',
    )

### Correlation Scatter + Regression Plot
Scatter plot with hue grouping alongside an OLS regression fit, side by side, for two chosen metrics.

In [ ]:
metrics_df_ivs.columns = (
    metrics_df_ivs.columns
    .str.replace('pore.', 'pore_', regex=False)
    .str.replace('throat.', 'throat_', regex=False)
)

In [ ]:
# check whether median pore volume tracks median throat diameter
ivs_results = plot_correlation_pair(
    metrics_df_ivs,
    colxn="pore_volume_median",
    colyn="throat_inscribed_diameter_median",
    n_colors=3
)

In [ ]:
df_ivs['sample_id']= df_ivs['filename'].str.split('_', expand=True)[1]
plt.figure(figsize=(8, 5))
sns.barplot(x=df_ivs['sample_id'], y=df_ivs['surface-area'], hue=df_ivs['sample_id'], palette='colorblind')
plt.xlabel("Sample ID")
plt.ylabel("Surface Area")
plt.title("IVS Surface Area by Sample ID")
plt.tight_layout();

# Villi Measurements

In [ ]:
# villi surface metrics there's no skeletonization or pore-network step here, just shape/surface-area measurements
df_villi = pd.read_csv("/path/to/quantification-notebooks/villi_metrics.csv")
df_villi.head()

### set filename as index

In [ ]:
df_villi['filename'].str.split('_', expand=True)[1]

In [ ]:
df_villi['sample_id']= df_villi['filename'].str.split('_', expand=True)[1]

### Quick EDA

Only `surface-area` is the surviving column here, so a simple bar plot is more informative than a full correlation plot done by `analyze_metrics`

In [ ]:
df_villi.columns

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=df_villi['sample_id'], y=df_villi['surface-area'], hue=df_villi['sample_id'], palette='colorblind')
plt.xlabel("Sample ID")
plt.ylabel("Surface Area")
plt.title("Villi Surface Area by Sample ID")
plt.tight_layout();